# WFCRL baselines

IPPO and MAPPO on Ablaincourt. The comparison uses load_coef = 0.1; the paper scripts default to 1.

In [1]:
from pathlib import Path
import os, subprocess, sys

ROOT = Path.cwd().resolve()
ENV_REPO = ROOT / "wfcrl-env"
BENCH = ROOT / "wfcrl-benchmark"
assert (ENV_REPO / "wfcrl" / "simple_env.py").exists()
assert (BENCH / "algos" / "baseline_ippo.py").exists()
print("Python:", sys.executable)
print("WFCRL:", ENV_REPO)
print("Algorithms:", BENCH)

Python: /Users/shreyansjain/Desktop/Wind-MARL/.venv/bin/python
WFCRL: /Users/shreyansjain/Desktop/Wind-MARL/wfcrl-env
Algorithms: /Users/shreyansjain/Desktop/Wind-MARL/wfcrl-benchmark


## Environment

In [2]:
sys.path.insert(0, str(ENV_REPO))
from wfcrl import environments as envs

env = envs.make("Dec_Ablaincourt_Floris", log=False, max_num_steps=151, load_coef=0.1)
env.reset(options={"wind_speed": 8.0, "wind_direction": 270.0})
agent = env.possible_agents[0]
print("agents:", env.possible_agents)
print("observation:", env.observe(agent))
print("action space:", env.action_space(agent))
env.close()

agents: ['turbine_1', 'turbine_2', 'turbine_3', 'turbine_4', 'turbine_5', 'turbine_6', 'turbine_7']
observation: OrderedDict({'yaw': np.float64(0.0), 'wind_speed': np.float64(7.973632994592287), 'wind_direction': np.float64(270.26410649314425)})
action space: {'yaw': Box(-5.0, 5.0, (1,), float32)}


In [3]:
sys.path.insert(0, str(BENCH / "algos"))
from extractors import VectorExtractor
from baseline_ippo import Agent as IPPOAgent
from baseline_mappo import Agent as MAPPOAgent, SharedCritic

local_obs = VectorExtractor(env.observation_space(agent)).space
local_action = VectorExtractor(env.action_space(agent)).space
global_obs = VectorExtractor(env.state_space).space
print("IPPO independent actor-critic:", IPPOAgent(local_obs, local_action, (64, 64)))
print("MAPPO local actor:", MAPPOAgent(local_obs, local_action, (64, 64)))
print("MAPPO shared global critic:", SharedCritic(global_obs, (64, 64)))

IPPO independent actor-critic: Agent(
  (critic): Sequential(
    (0): Sequential(
      (0): Linear(in_features=3, out_features=64, bias=True)
      (1): Tanh()
    )
    (1): Sequential(
      (0): Linear(in_features=64, out_features=64, bias=True)
      (1): Tanh()
    )
    (2): Linear(in_features=64, out_features=1, bias=True)
  )
  (actor): Sequential(
    (0): Sequential(
      (0): Linear(in_features=3, out_features=64, bias=True)
      (1): Tanh()
    )
    (1): Sequential(
      (0): Linear(in_features=64, out_features=64, bias=True)
      (1): Tanh()
    )
    (2): Linear(in_features=64, out_features=1, bias=True)
  )
)
MAPPO local actor: Agent(
  (actor): Sequential(
    (0): Sequential(
      (0): Linear(in_features=3, out_features=64, bias=True)
      (1): Tanh()
    )
    (1): Sequential(
      (0): Linear(in_features=64, out_features=64, bias=True)
      (1): Tanh()
    )
    (2): Linear(in_features=64, out_features=1, bias=True)
  )
)
MAPPO shared global critic: Shared

## Train

Use episode_length = 151 for 150 controlled steps; reset consumes one step. Set it to 150 for the authors' default.

In [4]:
ALGORITHMS = ("ippo", "mappo")
SEEDS = (1,)
SCENARIO = "windrose"
LOAD_COEF = 0.1  # Use 1.0 for the paper setting
TOTAL_TIMESTEPS = 100_000
EPISODE_LENGTH = 151  # 150 controlled steps after WFCRL reset

run_env = os.environ.copy()
run_env["MPLBACKEND"] = "Agg"
run_env["PYTHONPATH"] = str(ENV_REPO) + os.pathsep + run_env.get("PYTHONPATH", "")
run_env["MPLCONFIGDIR"] = str(ROOT / "__simul__" / "matplotlib")
Path(run_env["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

def train_baseline(algorithm, seed):
    assert algorithm in {"ippo", "mappo"}
    command = [sys.executable, f"algos/baseline_{algorithm}.py",
               "--env-id", "Dec_Ablaincourt_Floris", "--scenario", SCENARIO,
               "--wind-data", "data/smarteole.csv", "--seed", str(seed),
               "--load-coef", str(LOAD_COEF), "--episode-length", str(EPISODE_LENGTH),
               "--total-timesteps", str(TOTAL_TIMESTEPS),
               "--no-track", "--no-cuda"]
    print("Running:", " ".join(command), flush=True)
    subprocess.run(command, cwd=BENCH, env=run_env, check=True)

for algorithm in ALGORITHMS:
    for seed in SEEDS:
        train_baseline(algorithm, seed)

Running: /Users/shreyansjain/Desktop/Wind-MARL/.venv/bin/python algos/baseline_ippo.py --env-id Dec_Ablaincourt_Floris --scenario windrose --wind-data data/smarteole.csv --seed 1 --load-coef 0.1 --episode-length 151 --total-timesteps 100000 --no-track --no-cuda


Matplotlib is building the font cache; this may take a moment.
/Users/shreyansjain/Desktop/Wind-MARL/.venv/lib/python3.12/site-packages/tyro/_parsers.py:365: UserWarning: The field `wandb-entity` is annotated with type `<class 'str'>`, but the default value `None` has type `<class 'NoneType'>`. We'll try to handle this gracefully, but it may cause unexpected behavior.
  warnings.warn(message)
/Users/shreyansjain/Desktop/Wind-MARL/.venv/lib/python3.12/site-packages/tyro/_parsers.py:365: UserWarning: The field `pretrained-models` is annotated with type `<class 'str'>`, but the default value `None` has type `<class 'NoneType'>`. We'll try to handle this gracefully, but it may cause unexpected behavior.
  warnings.warn(message)


Evaluating at iteration 0
Evaluating on 25 episodes
Episode: 0 Score:  359.12761656706914
model saved to runs/Dec_Ablaincourt_Floris__baseline_ippo__windrose_1__1794149416/baseline_ippo.cleanrl_model
Evaluating at iteration 5
Evaluating on 25 episodes
Episode: 5 Score:  371.9545424818303
Evaluating at iteration 10
Evaluating on 25 episodes
Episode: 10 Score:  367.60711923833355
Evaluating at iteration 15
Evaluating on 25 episodes
Episode: 15 Score:  372.2345850767324
Evaluating at iteration 20
Evaluating on 25 episodes
Episode: 20 Score:  371.91710364186395
Evaluating at iteration 25
Evaluating on 25 episodes
Episode: 25 Score:  371.5680030742435
Evaluating at iteration 30
Evaluating on 25 episodes
Episode: 30 Score:  357.2829455139039
Evaluating at iteration 35
Evaluating on 25 episodes
Episode: 35 Score:  347.5778686240576
Evaluating at iteration 40
Evaluating on 25 episodes
Episode: 40 Score:  342.54331809110994
Evaluating at iteration 45
Evaluating on 25 episodes
Episode: 45 Score:

/Users/shreyansjain/Desktop/Wind-MARL/.venv/lib/python3.12/site-packages/tyro/_parsers.py:365: UserWarning: The field `wandb-entity` is annotated with type `<class 'str'>`, but the default value `None` has type `<class 'NoneType'>`. We'll try to handle this gracefully, but it may cause unexpected behavior.
  warnings.warn(message)
/Users/shreyansjain/Desktop/Wind-MARL/.venv/lib/python3.12/site-packages/tyro/_parsers.py:365: UserWarning: The field `pretrained-models` is annotated with type `<class 'str'>`, but the default value `None` has type `<class 'NoneType'>`. We'll try to handle this gracefully, but it may cause unexpected behavior.
  warnings.warn(message)


Evaluating at iteration 0
Evaluating on 25 episodes
Episode: 0 Score:  358.773955676067
model saved to runs/Dec_Ablaincourt_Floris__baseline_mappo__windrose_1__1794150208/baseline_mappo.cleanrl_model
Evaluating at iteration 5
Evaluating on 25 episodes
Episode: 5 Score:  354.3761991121799
Evaluating at iteration 10
Evaluating on 25 episodes
Episode: 10 Score:  355.7635920568123
Evaluating at iteration 15
Evaluating on 25 episodes
Episode: 15 Score:  354.2353680152552
Evaluating at iteration 20
Evaluating on 25 episodes
Episode: 20 Score:  355.77159929028926
Evaluating at iteration 25
Evaluating on 25 episodes
Episode: 25 Score:  347.1890015406053
Evaluating at iteration 30
Evaluating on 25 episodes
Episode: 30 Score:  349.17692583975537
Evaluating at iteration 35
Evaluating on 25 episodes
Episode: 35 Score:  348.02347905800315
Evaluating at iteration 40
Evaluating on 25 episodes
Episode: 40 Score:  340.79214877359135
Evaluating at iteration 45
Evaluating on 25 episodes
Episode: 45 Score

## Logs

In [6]:
%load_ext tensorboard
%tensorboard --logdir {BENCH / "runs"}

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


Reusing TensorBoard on port 6006 (pid 81710), started 0:00:51 ago. (Use '!kill 81710' to kill it.)